In [1]:
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent
df = pd.read_csv(BASE_DIR/'adult.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


# <span style="color:yellow">Handling '?'</span>

In [3]:
for col in df.columns:
    print(f'{col}: {(df[col] == '?').sum()}')
    
print(len(df))

for col in df.columns:
    if (df[col] == '?').any():
        if col == 'native_country':
            df[col] = df[col].replace('?', df[col].mode()[0])
        else:
            df = (df[df[col] != '?'])

df = df.reset_index(drop=True)

print(len(df))

age: 0
workclass: 1836
fnlwgt: 0
education: 0
education_num: 0
marital_status: 0
occupation: 1843
relationship: 0
race: 0
sex: 0
capital_gain: 0
capital_loss: 0
hours_per_week: 0
native_country: 583
income: 0
32561
30718


# <span style="color:yellow">Handling columns</span>

In [4]:
df['capital_net'] = df['capital_gain'] - df['capital_loss']

to_drop = [
    'fnlwgt', 'education',
    'capital_gain', 'capital_loss'
]

df = df.drop(columns=to_drop)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30718 entries, 0 to 30717
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             30718 non-null  int64 
 1   workclass       30718 non-null  object
 2   education_num   30718 non-null  int64 
 3   marital_status  30718 non-null  object
 4   occupation      30718 non-null  object
 5   relationship    30718 non-null  object
 6   race            30718 non-null  object
 7   sex             30718 non-null  object
 8   hours_per_week  30718 non-null  int64 
 9   native_country  30718 non-null  object
 10  income          30718 non-null  object
 11  capital_net     30718 non-null  int64 
dtypes: int64(4), object(8)
memory usage: 2.8+ MB


# <span style="color:yellow">Encoding</span>

In [5]:
df['income'] = df['income'].str.lower().str.strip().map({'<=50k': 0,
                                                       '>50k': 1})

obj = df.select_dtypes(include=['object'])
for col in obj:
    print(f'{col}: {df[col].nunique()}')

# frequency for native_country; one-hot for the rest
count = df['native_country'].value_counts()
df['native_country'] = df['native_country'].map(count)

one_hot = [col for col in obj if col != 'native_country']
df = pd.get_dummies(df, columns=one_hot, drop_first=True)


bools = df.select_dtypes(include=['bool'])
for col in bools:
    df[col] = df[col].astype(int)


df.info()

workclass: 7
marital_status: 7
occupation: 14
relationship: 6
race: 5
sex: 2
native_country: 41
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30718 entries, 0 to 30717
Data columns (total 41 columns):
 #   Column                                Non-Null Count  Dtype
---  ------                                --------------  -----
 0   age                                   30718 non-null  int64
 1   education_num                         30718 non-null  int64
 2   hours_per_week                        30718 non-null  int64
 3   native_country                        30718 non-null  int64
 4   income                                30718 non-null  int64
 5   capital_net                           30718 non-null  int64
 6   workclass_Local-gov                   30718 non-null  int32
 7   workclass_Private                     30718 non-null  int32
 8   workclass_Self-emp-inc                30718 non-null  int32
 9   workclass_Self-emp-not-inc            30718 non-null  int32
 10  workclass_

# <span style="color:yellow">Standardization</span>

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['income'])
y = df['income']

rawx_tr, rawx_ts, y_tr, y_ts = train_test_split(X, y,
                                         test_size=0.2,
                                         random_state=50)

my_scaler = StandardScaler()

x_tr = my_scaler.fit_transform(rawx_tr)
x_ts = my_scaler.transform(rawx_ts)

# <span style="color:orange">TRAINING :</span>

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score

### <span style="color:orange">Logistic Regression</span>

In [8]:
itr_vals = [400, 700, 1000]
c_vals = [0.02, 0.1, 1, 5]    # regularization strength
for itr in itr_vals:
    print(f'itr: {itr}')
    for c in c_vals:
        log_model = LogisticRegression(max_iter=itr,
                                       penalty='l2',
                                       C= c,
                                      random_state=3)
        
        log_model.fit(x_tr, y_tr)
        log_preds = log_model.predict(x_ts)

        acc = accuracy_score(y_ts, log_preds)
        prc = precision_score(y_ts, log_preds)
        rec = recall_score(y_ts, log_preds)

        print(f'C: {c} --> Acc: {acc:.5f} , Prc: {prc:.5f}, Rec: {rec:.5f}')

itr: 400
C: 0.02 --> Acc: 0.84212 , Prc: 0.73910, Rec: 0.58579
C: 0.1 --> Acc: 0.84212 , Prc: 0.73604, Rec: 0.59091
C: 1 --> Acc: 0.84163 , Prc: 0.73504, Rec: 0.58963
C: 5 --> Acc: 0.84147 , Prc: 0.73482, Rec: 0.58899
itr: 700
C: 0.02 --> Acc: 0.84212 , Prc: 0.73910, Rec: 0.58579
C: 0.1 --> Acc: 0.84212 , Prc: 0.73604, Rec: 0.59091
C: 1 --> Acc: 0.84163 , Prc: 0.73504, Rec: 0.58963
C: 5 --> Acc: 0.84147 , Prc: 0.73482, Rec: 0.58899
itr: 1000
C: 0.02 --> Acc: 0.84212 , Prc: 0.73910, Rec: 0.58579
C: 0.1 --> Acc: 0.84212 , Prc: 0.73604, Rec: 0.59091
C: 1 --> Acc: 0.84163 , Prc: 0.73504, Rec: 0.58963
C: 5 --> Acc: 0.84147 , Prc: 0.73482, Rec: 0.58899


### <span style="color:orange">KNN</span>

In [9]:
k_vals = range(3, 12, 2)
for k in  k_vals:
    knn_model = KNeighborsClassifier(n_neighbors=k, weights='distance', 
                                     metric='minkowski', p=2)
    
    knn_model.fit(x_tr, y_tr)
    knn_preds = knn_model.predict(x_ts)

    acc = accuracy_score(y_ts, knn_preds)
    prc = precision_score(y_ts, knn_preds)
    rec = recall_score(y_ts, knn_preds)

    print(f'K: {k} --> Acc: {acc:.5f} , Prc: {prc:.5f}, Rec: {rec:.5f}')

K: 3 --> Acc: 0.81396 , Prc: 0.65669, Rec: 0.56210
K: 5 --> Acc: 0.81999 , Prc: 0.67378, Rec: 0.56594
K: 7 --> Acc: 0.82747 , Prc: 0.69764, Rec: 0.56722
K: 9 --> Acc: 0.82780 , Prc: 0.70000, Rec: 0.56466
K: 11 --> Acc: 0.82780 , Prc: 0.69811, Rec: 0.56850


### <span style="color:orange">SVM</span>

In [10]:
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
c_vals = [0.1, 1, 5]    # regularization strength
for ker in kernels:
    print(f'kernel: {ker}')
    for c in c_vals:
        svm_model = SVC(kernel=ker, 
                        C=c,
                       random_state=20)
        
        svm_model.fit(x_tr, y_tr)
        svm_preds = svm_model.predict(x_ts)

        acc = accuracy_score(y_ts, svm_preds)
        prc = precision_score(y_ts, svm_preds)
        rec = recall_score(y_ts, svm_preds)

        print(f'C: {c} --> Acc: {acc:.5f} , Prc: {prc:.5f}, Rec: {rec:.5f}')

kernel: linear
C: 0.1 --> Acc: 0.84082 , Prc: 0.73856, Rec: 0.57875
C: 1 --> Acc: 0.84163 , Prc: 0.73924, Rec: 0.58259
C: 5 --> Acc: 0.84163 , Prc: 0.73924, Rec: 0.58259
kernel: rbf
C: 0.1 --> Acc: 0.83447 , Prc: 0.74840, Rec: 0.52561
C: 1 --> Acc: 0.84163 , Prc: 0.74852, Rec: 0.56786
C: 5 --> Acc: 0.84229 , Prc: 0.74444, Rec: 0.57810
kernel: poly
C: 0.1 --> Acc: 0.81999 , Prc: 0.75446, Rec: 0.43278
C: 1 --> Acc: 0.83317 , Prc: 0.73615, Rec: 0.53585
C: 5 --> Acc: 0.83561 , Prc: 0.72366, Rec: 0.57170
kernel: sigmoid
C: 0.1 --> Acc: 0.81624 , Prc: 0.67446, Rec: 0.53585
C: 1 --> Acc: 0.78369 , Prc: 0.57920, Rec: 0.54545
C: 5 --> Acc: 0.77962 , Prc: 0.56869, Rec: 0.55122


### <span style="color:orange">Naive Bayes</span>

In [11]:
naive_model = GaussianNB(priors=None,    # Prior probabilities of each class learned from the data
                        var_smoothing=1e-8)

naive_model.fit(x_tr, y_tr)
naive_preds = naive_model.predict(x_ts)

acc = accuracy_score(y_ts, naive_preds)
prc = precision_score(y_ts, naive_preds)
rec = recall_score(y_ts, naive_preds)

print(f'Acc: {acc:.5f} , Prc: {prc:.5f}, Rec: {rec:.5f}')

Acc: 0.58073 , Prc: 0.37280, Rec: 0.95134


### <span style="color:orange">Neural Network</span>

In [12]:
actives = ['relu', 'tanh', 'logistic']
alpha_vals = [0.0001, 0.001, 0.01]    # regularization strength
for func in actives:
    print(f'Activation: {func}')
    for al in alpha_vals:
        net_model = MLPClassifier(hidden_layer_sizes= (64,),
                                activation=func,
                                solver='adam',
                                alpha=al,
                                random_state=42)
    
        net_model.fit(x_tr, y_tr)
        net_preds = net_model.predict(x_ts)
    
        acc = accuracy_score(y_ts, net_preds)
        prc = precision_score(y_ts, net_preds)
        rec = recall_score(y_ts, net_preds)
    
        print(f'alpha: {al} --> Acc: {acc:.5f} , Prc: {prc:.5f}, Rec: {rec:.5f}')

Activation: relu


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


alpha: 0.0001 --> Acc: 0.84619 , Prc: 0.73822, Rec: 0.61204


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


alpha: 0.001 --> Acc: 0.84945 , Prc: 0.74557, Rec: 0.61908


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


... (8 outputs left out)

alpha: 0.0001 --> Acc: 0.84880 , Prc: 0.74980, Rec: 0.60819


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


alpha: 0.001 --> Acc: 0.84782 , Prc: 0.74861, Rec: 0.60435
alpha: 0.01 --> Acc: 0.84831 , Prc: 0.74803, Rec: 0.60819


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


## <span style="color:yellow">Final Model:</span>
### <span style="color:yellow">Neural Network, ReLU, alpha= 0.001</span>

In [13]:
final_model = MLPClassifier(hidden_layer_sizes= (64,),
                            activation='relu',
                            solver='adam',
                            alpha=0.001,
                            random_state=42)
    
final_model.fit(x_tr, y_tr)
final_preds = final_model.predict(x_ts)
    
acc = accuracy_score(y_ts, final_preds)
prc = precision_score(y_ts, final_preds)
rec = recall_score(y_ts, final_preds)
    
print(f'Acc: {acc:.5f} , Prc: {prc:.5f}, Rec: {rec:.5f}')

Acc: 0.84945 , Prc: 0.74557, Rec: 0.61908


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


# <span style="color:yellow">Exporting</span>

In [14]:
import joblib

joblib.dump(final_model, BASE_DIR/'Experimental nb'/'NN_model.pkl')
joblib.dump(my_scaler, BASE_DIR/'Experimental nb'/'scaler.pkl')

print('Files "scaler.pkl" & "NN_model.pkl" exported.')

Files "scaler.pkl" & "NN_model.pkl" exported.
